# Comparison Thresholds

## What is this notebook doing?
This notebook performs a full pairwise comparison across all videos in the dataset for every fingerprinting method explored in notebooks 02 and 03. Ground truth positive pairs are derived automatically from the filename convention `creator_videodescription_platform.mp4` — any two videos sharing the same `creator_videodescription` prefix are considered true matches.

## Methods compared
Four fingerprinting methods are evaluated, each producing two similarity scores:
- **pHash** — min_diff and mean top-k (Hamming distance)
- **dHash** — min_diff and mean top-k (Hamming distance)
- **HSV Color Histogram** — min_diff and mean top-k (cosine distance)
- **Chromaprint** — min_diff and mean top-k (normalized Hamming distance)

This yields 8 score distributions. For each, the threshold is set to the worst (highest) score among all true positive pairs, guaranteeing no false negatives. Distributions of positive and negative pair scores are plotted to visualize method separability.

## Efficiency
A single full pairwise pass is performed per method. Both min_diff and mean top-k are computed simultaneously from the same pass, giving all 8 distributions from 4 passes total.

In [ ]:
import os
import re
import json
import itertools
import numpy as np
import imagehash
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine as cosine_dist

# Config
TOP_K = 5
VISUAL_JSON = "visual_fingerprints.json"   # from notebook 02
AUDIO_JSON  = "audio_fingerprints.json"    # from notebook 03

# Regex: extract everything before the last _platform suffix
# e.g. 'scottkress_halloween_dl.mp4' -> 'scottkress_halloween'
PREFIX_PATTERN = re.compile(r'^(.+)_[^_]+$')

In [ ]:
# --- Ground truth generation ---

def extract_prefix(filename):
    """Strip extension and platform suffix to get match group."""
    name = os.path.splitext(filename)[0]
    match = PREFIX_PATTERN.match(name)
    return match.group(1) if match else name

def build_ground_truth(video_names):
    """
    Returns a set of frozensets, each containing a true positive pair.
    Two videos are a positive pair if they share the same prefix.
    """
    prefix_to_videos = {}
    for name in video_names:
        prefix = extract_prefix(name)
        prefix_to_videos.setdefault(prefix, []).append(name)

    positive_pairs = set()
    for videos in prefix_to_videos.values():
        for a, b in itertools.combinations(videos, 2):
            positive_pairs.add(frozenset([a, b]))

    return positive_pairs, prefix_to_videos

# Load video names from visual JSON (audio JSON should have the same set)
with open(VISUAL_JSON, "r") as f:
    visual_data = json.load(f)

video_names = list(visual_data.keys())
positive_pairs, prefix_groups = build_ground_truth(video_names)
all_pairs = list(itertools.combinations(video_names, 2))

print(f"Total videos:         {len(video_names)}")
print(f"Total pairs:          {len(all_pairs)}")
print(f"True positive pairs:  {len(positive_pairs)}")
print(f"True negative pairs:  {len(all_pairs) - len(positive_pairs)}")
print(f"\nMatch groups:")
for prefix, videos in prefix_groups.items():
    print(f"  {prefix}: {videos}")

In [ ]:
# --- Load fingerprints ---

with open(AUDIO_JSON, "r") as f:
    audio_data = json.load(f)

def get_visual_chunks(video_name, method):
    chunks = visual_data[video_name]["chunks"]
    result = []
    for chunk in chunks:
        frames = chunk["frames"]
        if not frames:
            continue
        if method in ("phash", "dhash"):
            result.append(imagehash.hex_to_hash(frames[0][method]))
        elif method == "color_histogram":
            vecs = np.array([f["color_histogram"] for f in frames])
            result.append(np.mean(vecs, axis=0))
    return result

def get_audio_chunks(video_name):
    return [
        np.array(chunk["fingerprint"], dtype=np.int32)
        for chunk in audio_data[video_name]["chunks"]
    ]

def visual_distance(c1, c2, method):
    if method in ("phash", "dhash"):
        return abs(c1 - c2)
    elif method == "color_histogram":
        return cosine_dist(c1, c2)

def audio_distance(fp1, fp2):
    min_len = min(len(fp1), len(fp2))
    if min_len == 0:
        return 1.0
    xor = np.bitwise_xor(fp1[:min_len], fp2[:min_len])
    bits_diff = sum(bin(x).count('1') for x in xor)
    return bits_diff / (min_len * 32)

print("Fingerprints loaded.")

In [ ]:
# --- Full pairwise comparison (single pass per method) ---
# For each pair, compute both min_diff and mean_top_k simultaneously.
# Results stored as: pairwise_scores[method][frozenset(a, b)] = {min_diff, mean_top_k}

VISUAL_METHODS = ["phash", "dhash", "color_histogram"]
pairwise_scores = {m: {} for m in VISUAL_METHODS + ["chromaprint"]}

total_pairs = len(all_pairs)

# Visual methods
for method in VISUAL_METHODS:
    print(f"Running pairwise comparison: {method}...")
    # Pre-load all chunk fingerprints
    all_chunks = {v: get_visual_chunks(v, method) for v in video_names}

    for i, (a, b) in enumerate(all_pairs):
        chunks_a = all_chunks[a]
        chunks_b = all_chunks[b]

        dists = [
            visual_distance(c1, c2, method)
            for c1 in chunks_a
            for c2 in chunks_b
        ]

        if not dists:
            continue

        dists_sorted = sorted(dists)
        pairwise_scores[method][frozenset([a, b])] = {
            "min_diff": dists_sorted[0],
            "mean_top_k": float(np.mean(dists_sorted[:TOP_K]))
        }

        if (i + 1) % 10 == 0 or (i + 1) == total_pairs:
            print(f"  {i + 1}/{total_pairs} pairs")

    print(f"  {method} complete.")

# Audio (Chromaprint)
print("Running pairwise comparison: chromaprint...")
all_audio_chunks = {v: get_audio_chunks(v) for v in video_names}

for i, (a, b) in enumerate(all_pairs):
    chunks_a = all_audio_chunks[a]
    chunks_b = all_audio_chunks[b]

    dists = [
        audio_distance(c1, c2)
        for c1 in chunks_a
        for c2 in chunks_b
    ]

    if not dists:
        continue

    dists_sorted = sorted(dists)
    pairwise_scores["chromaprint"][frozenset([a, b])] = {
        "min_diff": dists_sorted[0],
        "mean_top_k": float(np.mean(dists_sorted[:TOP_K]))
    }

    if (i + 1) % 10 == 0 or (i + 1) == total_pairs:
        print(f"  {i + 1}/{total_pairs} pairs")

print("All pairwise comparisons complete.")

In [ ]:
# --- Threshold distributions ---
# 8 plots: 4 methods x 2 scores
# Each plot shows positive pair scores vs negative pair scores
# with the max positive score marked as the zero-FN threshold

ALL_METHODS = VISUAL_METHODS + ["chromaprint"]
SCORE_KEYS = ["min_diff", "mean_top_k"]

# Collect auto thresholds while plotting
auto_thresholds = {m: {} for m in ALL_METHODS}

fig, axes = plt.subplots(len(ALL_METHODS), 2, figsize=(14, 5 * len(ALL_METHODS)))

for row_idx, method in enumerate(ALL_METHODS):
    scores = pairwise_scores[method]

    for col_idx, score_key in enumerate(SCORE_KEYS):
        pos_scores = [
            scores[pair][score_key]
            for pair in positive_pairs
            if pair in scores
        ]
        neg_scores = [
            scores[pair][score_key]
            for pair in scores
            if pair not in positive_pairs
        ]

        threshold = max(pos_scores) if pos_scores else None
        auto_thresholds[method][score_key] = threshold

        ax = axes[row_idx, col_idx]
        ax.hist(neg_scores, bins=30, alpha=0.6, color="steelblue", label="Negative pairs")
        ax.hist(pos_scores, bins=30, alpha=0.7, color="salmon", label="Positive pairs")

        if threshold is not None:
            ax.axvline(threshold, color="red", linestyle="--", linewidth=1.5,
                       label=f"Threshold: {threshold:.4f}")

        ax.set_title(f"{method} — {score_key}", fontsize=10)
        ax.set_xlabel("Distance")
        ax.set_ylabel("Count")
        ax.legend(fontsize=8)

plt.suptitle("Score Distributions: Positive vs Negative Pairs", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("\nAuto-computed thresholds (max positive pair score):")
print(f"{'Method':<20} {'min_diff':>12} {'mean_top_k':>14}")
print("-" * 48)
for method in ALL_METHODS:
    md = auto_thresholds[method]['min_diff']
    mk = auto_thresholds[method]['mean_top_k']
    print(f"{method:<20} {md:>12.6f} {mk:>14.6f}")

In [ ]:
# --- Configurable thresholds ---
# Pre-populated from auto-computed values above.
# Adjust any value here and re-run from Cell 8 to evaluate performance.

THRESHOLDS = {
    "phash": {
        "min_diff":   auto_thresholds["phash"]["min_diff"],
        "mean_top_k": auto_thresholds["phash"]["mean_top_k"]
    },
    "dhash": {
        "min_diff":   auto_thresholds["dhash"]["min_diff"],
        "mean_top_k": auto_thresholds["dhash"]["mean_top_k"]
    },
    "color_histogram": {
        "min_diff":   auto_thresholds["color_histogram"]["min_diff"],
        "mean_top_k": auto_thresholds["color_histogram"]["mean_top_k"]
    },
    "chromaprint": {
        "min_diff":   auto_thresholds["chromaprint"]["min_diff"],
        "mean_top_k": auto_thresholds["chromaprint"]["mean_top_k"]
    }
}

In [ ]:
# --- Performance summary ---
# Precision, recall, and F1 at each configured threshold

def evaluate(scores, positive_pairs, threshold, score_key):
    tp = sum(1 for p, s in scores.items() if p in positive_pairs and s[score_key] <= threshold)
    fp = sum(1 for p, s in scores.items() if p not in positive_pairs and s[score_key] <= threshold)
    tn = sum(1 for p, s in scores.items() if p not in positive_pairs and s[score_key] > threshold)
    fn = sum(1 for p, s in scores.items() if p in positive_pairs and s[score_key] > threshold)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return {"tp": tp, "fp": fp, "tn": tn, "fn": fn,
            "precision": precision, "recall": recall, "f1": f1}

print(f"{'Method':<20} {'Score':<12} {'Threshold':>10} {'TP':>4} {'FP':>4} {'TN':>4} {'FN':>4} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("-" * 96)

for method in ALL_METHODS:
    for score_key in SCORE_KEYS:
        threshold = THRESHOLDS[method][score_key]
        m = evaluate(pairwise_scores[method], positive_pairs, threshold, score_key)
        print(
            f"{method:<20} {score_key:<12} {threshold:>10.4f} "
            f"{m['tp']:>4} {m['fp']:>4} {m['tn']:>4} {m['fn']:>4} "
            f"{m['precision']:>10.4f} {m['recall']:>8.4f} {m['f1']:>8.4f}"
        )

## Conclusion

**Thresholds chosen:**

| Method | Score | Threshold |
|---|---|---|
| phash | min_diff | |
| phash | mean_top_k | |
| dhash | min_diff | |
| dhash | mean_top_k | |
| color_histogram | min_diff | |
| color_histogram | mean_top_k | |
| chromaprint | min_diff | |
| chromaprint | mean_top_k | |

**Reasoning:**